# LLM API と Ollama - OpenAI以外の選択肢

_重要: APIそのものや、PC・Macでの環境変数についてあまり馴染みがない場合は、このガイドに進む前にガイド4「技術的基礎」のAPIセクション（ガイド4のトピック3と5）を復習してください。_

## OpenAI以外のモデルを使うための重要な前提知識 - まずこれを読んでください！

このコースでは、地球上で最も強力なLLMに接続するためにAPIを使用します。

OpenAI、Anthropic、Google、DeepSeekといったこれらのLLMを提供する企業は、Webエンドポイントを構築しています。あなたはWebアドレスにHTTPリクエストを送り、プロンプトに関するすべての情報を渡すことで、それらのモデルを呼び出します。

しかし、APIを呼び出すたびにHTTPリクエストを自分で組み立てるのは大変です。

これを簡単にするために、OpenAIのチームは「Pythonクライアントライブラリ」と呼ばれるPythonユーティリティを書きました。これはHTTP呼び出しをラップしてくれるものです。つまり、あなたはPythonコードを書き、それがWebを呼び出してくれるのです。

これこそが `openai` ライブラリの正体です。

### `openai` というPythonクライアントライブラリとは何か

これは：
- 軽量なPythonユーティリティである
- PythonのリクエストをHTTP呼び出しに変換する
- HTTP呼び出しから返ってきた結果をPythonオブジェクトに変換する

### これではないもの

- 実際にLarge Language Modelを実行するコードは一切含まれていません！GPTのコードなどはなく、単にWebリクエストを行うだけです
- 科学計算用のコードもなく、OpenAI向けに特別な処理をしているわけでもありません

### 使い方

```python
# OpenAIへのWeb呼び出しを行うためのOpenAI Pythonクライアントを作成
openai = OpenAI()

# 呼び出しを実行
response = openai.chat.completions.create(model="gpt-4.1-mini", messages=[{"role":"user", "content": "what is 2+2?"}])

# 結果を表示
print(response.choices[0].message.content)
```

### これは何をしているのか

Pythonの呼び出し `openai.chat.completions.create()` を実行すると、  
単純に次のURLへWebリクエストが送られます: `https://api.openai.com/v1/chat/completions`  
そして、そのレスポンスをPythonオブジェクトに変換しているだけです。

それだけです。

[直接Web HTTP呼び出しを行う場合](https://platform.openai.com/docs/guides/text?api-mode=chat&lang=curl)のAPIドキュメントはこちら。  
[Pythonクライアントライブラリ](https://platform.openai.com/docs/guides/text?api-mode=chat&lang=python)を使う場合の同じAPIドキュメントはこちら。

## この前提を踏まえて - 他のLLMをどう使うのか？

実はこれ、とても簡単なのです！

他の主要なLLMも、OpenAI互換のAPIエンドポイントを持っています。

そこでOpenAIは、みんなのためにひと工夫してくれました。「Pythonをウェブリクエストに変換するこのユーティリティは、誰でも使っていいですよ。呼び出し先を `https://api.openai/com/v1` から、あなたが指定する任意のWebアドレスに変更できるようにします」というわけです。

つまり、次のようにすることで、OpenAI製ではないモデルの呼び出しにもOpenAIのユーティリティを使うことができます。

`not_actually_openai = OpenAI(base_url="https://somewhere.completely.different/", api_key="another_providers_key")`

このOpenAIのコードは、あくまでエンドポイントへHTTP呼び出しを行うためのユーティリティに過ぎない、ということを理解しておくのが重要です。だからこそ、OpenAIチームのコードを使っていても、OpenAI以外のモデルを呼び出すことができるのです。

主要なプロバイダーが提供している、OpenAI互換のエンドポイントを以下にまとめました。Ollamaをローカルで使う場合も含まれています。Ollamaはあなたのローカルマシン上にエンドポイントを提供していますが、これもOpenAI互換にしてくれているので非常に便利です。

```python
ANTHROPIC_BASE_URL = "https://api.anthropic.com/v1/"
DEEPSEEK_BASE_URL = "https://api.deepseek.com/v1"
GEMINI_BASE_URL = "https://generativelanguage.googleapis.com/v1beta/openai/"
GROK_BASE_URL = "https://api.x.ai/v1"
GROQ_BASE_URL = "https://api.groq.com/openai/v1"
OPENROUTER_BASE_URL = "https://openrouter.ai/api/v1"
OLLAMA_BASE_URL = "http://localhost:11434/v1"
```

## Gemini、DeepSeek、Ollama、OpenRouterの例

### 例1: OpenAIの代わりにGeminiを使う

1. Google Studioにアクセスしてアカウントを作成します: https://aistudio.google.com/  
2. あなたのキーを `.env` に `GOOGLE_API_KEY` として追加します。  
3. さらに同じキーを `GEMINI_API_KEY` としても追加してください - これは後で役に立ちます。

その上で：

```python
import os
from openai import OpenAI
from dotenv import load_dotenv
load_dotenv(override=True)

GEMINI_BASE_URL = "https://generativelanguage.googleapis.com/v1beta/openai/"
google_api_key = os.getenv("GOOGLE_API_KEY")
gemini = OpenAI(base_url=GEMINI_BASE_URL, api_key=google_api_key)
response = gemini.chat.completions.create(model="gemini-2.5-flash-lite", messages=[{"role":"user", "content": "what is 2+2?"}])
print(response.choices[0].message.content)
```

### 例2: OpenAIの代わりにDeepSeek APIを使う（安価で、最初に2ドルだけ）

1. DeepSeek APIにアクセスしてアカウントを作成します: https://platform.deepseek.com/  
2. 最低2ドルの初期チャージが必要です。  
3. あなたのキーを `.env` に `DEEPSEEK_API_KEY` として追加します。  

その上で：

```python
import os
from openai import OpenAI
from dotenv import load_dotenv
load_dotenv(override=True)

DEEPSEEK_BASE_URL = "https://api.deepseek.com/v1"
deepseek_api_key = os.getenv("DEEPSEEK_API_KEY")
deepseek = OpenAI(base_url=DEEPSEEK_BASE_URL, api_key=deepseek_api_key)
response = deepseek.chat.completions.create(model="deepseek-chat", messages=[{"role":"user", "content": "what is 2+2?"}])
print(response.choices[0].message.content)
```

### 例3: OpenAIの代わりに無料でローカルに使えるOllamaを使う

Ollamaを使うとモデルをローカルで実行できます。あなたのマシン上にOpenAI互換のAPIを提供してくれます。  
Ollamaにはキーは不要です。あなたのクレジットカード情報を持つ第三者も存在しないので、いかなる種類のキーも必要ありません。

1. Ollamaを初めて使う場合は、こちらの手順に従ってインストールしてください: https://ollama.com  
2. その後、Cursorのターミナルで `ollama run llama3.2` を実行すると、Llama 3.2とチャットできます。  
注意: llama3.3やllama4は使わないでください - これらは家庭用コンピューター向けには設計されていない巨大なモデルです！ディスクをいっぱいにしてしまいます。  

その上で：

```python
!ollama pull llama3.2

from openai import OpenAI

OLLAMA_BASE_URL = "http://localhost:11434/v1"
ollama = OpenAI(base_url=OLLAMA_BASE_URL, api_key="anything")
response = ollama.chat.completions.create(model="llama3.2", messages=[{"role":"user", "content": "what is 2+2?"}])
print(response.choices[0].message.content)
```

### 例4: OpenAIの代わりに、より簡単な課金プロセスを持つ人気サービス[OpenRouter](https://openrouter.ai)を使う

OpenRouterは非常に便利です。多くのモデルへの無料アクセスを提供してくれますし、有料モデルへも少額の前払いで簡単にアクセスできます。

1. https://openrouter.ai でサインアップします。
2. 必要に応じて最低限の前払い残高を追加します。
3. あなたのキーを `.env` ファイルに `OPENROUTER_API_KEY` として追加します。

その上で：

```python
import os
from openai import OpenAI
from dotenv import load_dotenv
load_dotenv(override=True)

OPENROUTER_BASE_URL = "https://openrouter.ai/api/v1"
openrouter_api_key = os.getenv("OPENROUTER_API_KEY")
openrouter = OpenAI(base_url=OPENROUTER_BASE_URL, api_key=openrouter_api_key)
response = openrouter.chat.completions.create(model="openai/gpt-4.1-nano", messages=[{"role":"user", "content": "what is 2+2?"}])
print(response.choices[0].message.content)
```


### Agent Frameworksで異なるAPIプロバイダーを使う

Agent Frameworksを使うと、こうしたプロバイダー間の切り替えが簡単に行えます。コースの中で、いつでもLLMを切り替えて別のものを選ぶことができます。それぞれについての補足は以下にあります。OpenAI Agents SDKについては、このノートブックの後のセクションを参照してください。CrewAIについては、コースの中で扱いますが、簡単です。LiteLLMが期待するモデルへのフルパスを使うだけです。

## APIのコスト

各API呼び出しのコストは実際には非常に低く、このコースで使うほとんどの呼び出しは1セントの何分の一にも満たない金額です。

しかし、以下の点は非常に重要なので注意してください。

1. 複雑なAgenticプロジェクトでは、多くのLLM呼び出し（おそらく20～30回）が発生することがあり、それが積み重なっていきます。上限を設定し、使用状況を監視することが重要です。

2. Agentic AIでは、Agentがループに陥ったり、意図した以上の処理を実行してしまうリスクがあります。API使用状況を監視し、自分が納得できる以上の予算を設定しないようにしてください。一部のAPIには、あなたのカードに自動的にチャージされる「自動リフィル」設定がありますが、これは無効にしておくことを強くお勧めします。

3. あなたが納得できる範囲でのみお金を使うようにしてください。無料の代替手段としてOllamaがあり、必要であればこれを代わりに使うことができます。DeepSeek、Gemini 2.5 Flash、gpt-4.1-nanoは、かなり安価です。

これらのLLM呼び出しは通常、何兆回にも及ぶ浮動小数点演算を伴うということを心に留めておいてください - 誰かがその電気代を払っているのです！

### Ollama: 有料APIの無料代替手段（ただしllamaのバージョンに関する警告を必ずご覧ください）

Ollamaは、あなたのマシン上でローカルに動作するプロダクトです。オープンソースのモデルを実行でき、あなたのコンピューター上にOpenAI互換のAPIエンドポイントを提供します。

まず、以下にアクセスしてOllamaをダウンロードしてください。
https://ollama.com

次に、Cursorのターミナル（View メニュー >> Terminal）から、以下のコマンドを実行してモデルをダウンロードします。

```shell
ollama pull llama3.2
```

警告: llama3.3やllama4を使わないよう注意してください - これらはずっと大規模なモデルで、家庭用コンピューターには適していません。

さて、これで、次のようなコードがある場合：  
`openai = OpenAI()`  
これをそのまま次のように置き換えることができます。  
`openai = OpenAI(base_url='http://localhost:11434/v1', api_key='ollama')`  
そして、**gpt-4o-mini** のようなモデル名も **llama3.2** に置き換えます。  

この場合、`.env` ファイルに何も書き込む必要はありません。Ollamaを使う場合、すべてがあなたのコンピューター上で動作します。クラウド上の第三者を呼び出しているわけではなく、誰もあなたのクレジットカード情報を持っていないので、シークレットキーは一切不要です！上のコードにある `api_key='ollama'` は、OpenAIクライアントライブラリがapi_keyの指定を必須としているために書いているだけで、その値自体はOllamaによって無視されます。

以下は完全な例です。

```python
# これはコンピューター上で一度だけ実行する必要があります
!ollama pull llama3.2

from openai import OpenAI
MODEL = "llama3.2"
openai = OpenAI(base_url="http://localhost:11434/v1", api_key="ollama")

response = openai.chat.completions.create(
 model=MODEL,
 messages=[{"role": "user", "content": "What is 2 + 2?"}]
)

print(response.choices[0].message.content)
```

いずれのAgent Frameworksの中でOllamaを使う場合も、これと似たような変更が必要になります - 具体的な例はgoogleで検索するか、私に聞いてみてください。

### OpenRouter: OpenAIやその他のモデルへの便利なゲートウェイプラットフォーム

OpenRouterは、OpenAIを含む幅広いLLMに接続できるようにしてくれる第三者サービスです。

より簡易な課金プロセスを持つことで知られており、アメリカ国外の一部の国にとっては、より使いやすい場合があります。

まず、彼らのウェブサイトをチェックしてください。  
https://openrouter.ai/

次に、彼らのクイックスタートを見てみましょう。  
https://openrouter.ai/docs/quickstart

そして、あなたのキーを`.env`ファイルに追加します。  
```shell
OPENROUTER_API_KEY=sk-or....
```

さて、これで、次のようなコードがある場合：  
```python
MODEL = "gpt-4o-mini"
openai = OpenAI()
```

これを次のようなコードに置き換えることができます。

```python
MODEL = "openai/gpt-4o-mini"
openrouter_api_key = os.getenv("OPENROUTER_API_KEY")
openai = OpenAI(base_url="https://openrouter.ai/api/v1", api_key=openrouter_api_key)

response = openai.chat.completions.create(
 model=MODEL,
 messages=[{"role": "user", "content": "What is 2 + 2?"}]
)

print(response.choices[0].message.content)
```

いずれのAgent Frameworksの中でOpenRouterを使う場合も、これと似たような変更が必要になります - 具体的な例はgoogleで検索するか、私に聞いてみてください。

## OpenAI Agents SDK - 特別な手順

OpenAI Agents SDK（第2週と第6週）では、OpenAI自身が提供するモデルを使うのが特に簡単です。単にモデル名を渡すだけです。

`agent = Agent(name="Jokester", instructions="You are a joke teller", model="gpt-4o-mini")`

OpenAI互換のAPIを持つ他のプロバイダーに置き換えることもできます。次の3ステップで行います。

```python
DEEPSEEK_BASE_URL = "https://api.deepseek.com/v1"
deepseek_client = AsyncOpenAI(base_url=DEEPSEEK_BASE_URL, api_key=deepseek_api_key)
deepseek_model = OpenAIChatCompletionsModel(model="deepseek-chat", openai_client=deepseek_client)
```

そして、Agentを作成する際に、このモデルを渡すだけです。

`agent = Agent(name="Jokester", instructions="You are a joke teller", model=deepseek_model)`

他のOpenAI互換APIについても、同じ3ステップの手順で同様に対応できます。

```python
# 追加のインポート
from agents import OpenAIChatCompletionsModel
from openai import AsyncOpenAI

# ステップ1: プロバイダーがOpenAI互換APIを提供しているベースURLエンドポイントを指定する
GEMINI_BASE_URL = "https://generativelanguage.googleapis.com/v1beta/openai/"
GROK_BASE_URL = "https://api.x.ai/v1"
GROQ_BASE_URL = "https://api.groq.com/openai/v1"
OPENROUTER_BASE_URL = "https://openrouter.ai/api/v1"
OLLAMA_BASE_URL = "http://localhost:11434/v1"

# ステップ2: そのエンドポイント用のAsyncOpenAIオブジェクトを作成する
gemini_client = AsyncOpenAI(base_url=GEMINI_BASE_URL, api_key=google_api_key)
grok_client = AsyncOpenAI(base_url=GROK_BASE_URL, api_key=grok_api_key)
groq_client = AsyncOpenAI(base_url=GROQ_BASE_URL, api_key=groq_api_key)
openrouter_client = AsyncOpenAI(base_url=OPENROUTER_BASE_URL, api_key=openrouter_api_key)
ollama_client = AsyncOpenAI(base_url=OLLAMA_BASE_URL, api_key="ollama")

# ステップ3: Agentを作成する際に渡すモデルオブジェクトを作成する
gemini_model = OpenAIChatCompletionsModel(model="gemini-2.5-flash", openai_client=gemini_client)
grok_3_model = OpenAIChatCompletionsModel(model="grok-3-mini-beta", openai_client=openrouter_client)
llama3_3_model = OpenAIChatCompletionsModel(model="llama-3.3-70b-versatile", openai_client=groq_client)
grok_3_via_openrouter_model = OpenAIChatCompletionsModel(model="x-ai/grok-3-mini-beta", openai_client=openrouter_client)
llama_3_2_local_model = OpenAIChatCompletionsModel(model="llama3.2", openai_client=ollama_client)
```

### OpenAI Agents SDKでAzureを使う方法

こちらの手順を参照してください。  
https://techcommunity.microsoft.com/blog/azure-ai-services-blog/use-azure-openai-and-apim-with-the-openai-agents-sdk/4392537

例えば、次のようになります。
```python
from openai import AsyncAzureOpenAI
from agents import set_default_openai_client
from dotenv import load_dotenv
import os
 
# 環境変数を読み込む
load_dotenv(override=True)
 
# Azure OpenAIを使ってOpenAIクライアントを作成
openai_client = AsyncAzureOpenAI(
    api_key=os.getenv("AZURE_OPENAI_API_KEY"),
    api_version=os.getenv("AZURE_OPENAI_API_VERSION"),
    azure_endpoint=os.getenv("AZURE_OPENAI_ENDPOINT"),
    azure_deployment=os.getenv("AZURE_OPENAI_DEPLOYMENT")
)
 
# Agents SDKのデフォルトOpenAIクライアントとして設定
set_default_openai_client(openai_client)
```

## CrewAIの設定

こちらはCrewのLLM接続に関するドキュメントで、すべてのモデルで使用するモデル名が記載されています。受講生のSadan S.さんが指摘してくれた通り（ありがとうございます！）、Googleについては、`GOOGLE_API_KEY`ではなく`GEMINI_API_KEY`という環境変数を使う必要があることを知っておく価値があります。

https://docs.crewai.com/concepts/llms

また、詳細情報付きのチュートリアルはこちらです。

https://docs.crewai.com/how-to/llm-connections

## LangGraphの設定

LangGraphでOllamaを使う方法（他のモデルについても同様の手順に従ってください）：  
https://python.langchain.com/docs/integrations/chat/ollama/#installation

まず、パッケージを追加します。  
`uv add langchain-ollama`

次に、ラボの中で以下のように置き換えます。  
```python
from langchain_ollama import ChatOllama
# llm = ChatOpenAI(model="gpt-4o-mini")
llm = ChatOllama(model="gemma3:4b")
```

そして、当然ながら事前に `!ollama pull gemma3:4b`（または使用するモデル）を実行しておいてください。

これを追加してくれたMiroslav P.さん、そして質問してくれたArvin F.さんに感謝します！

## その他のモデルでLangGraphを使う

上記と同じレシピに従ってください。ただし、こちらにあるいずれのモデルも使用できます。  
https://python.langchain.com/docs/integrations/chat/



## 第5週 Agent Frameworks

第5週では、Google ADK（A2Aを含む）、AWS Strands、Pydantic AI、Microsoft Agent Framework、Agno、Mastraといった複数のagent frameworksを巡ります。それぞれのフレームワークには、使用するモデルやプロバイダーを設定するための独自の方法があります。

別のモデルやプロバイダーへの切り替えの具体的な方法については、`5_agent_frameworks`フォルダの中にある、その日のフォルダ内の`SWAP_AI.md`ファイルを参照してください。

## 心に留めておくべきこと

1. Ollamaを使ってモデルをローカルで実行したい場合、より小さなモデルは、より高度なプロジェクトに苦戦することがあるでしょう。異なるサイズや能力のモデルを試してみる必要があり、うまく動くものを見つけるにはかなりの根気が必要かもしれません。私たちのプロジェクトの中には、llama3.2にとって難しすぎるものがいくつかあると想定しています。代わりに、openrouter.aiの無料モデルや、DeepSeekのようなほぼ無料に近い非常に安価なモデルを検討してみてください。

2. チャットモデルは、推論モデルよりも良い結果を出すことがよくあります。というのも、推論モデルは一部のタスクを「考えすぎて」しまうことがあるからです。実際に試してみることが重要です。大きいことが必ずしも良いことではありません…

3. 少し紛らわしいのですが、似たような名前の2つの異なるプロバイダーが存在します。  
- Grokは、Elon Musk氏のXによるLLMです
- Groqは、オープンソースモデルを高速に推論するためのプラットフォームです

ある受講生が私に指摘してくれましたが、「Groq」の方が先に存在していたそうです！
